# TikHarm - Inference Demo (fine-tuned Swin3D-T + CNN14 fusion)

Runs the fine-tuned model saved as `best_finetuned_32frames.pt` on the **test split**, then exports
**20 correctly classified** and **10 incorrectly classified** samples as a downloadable zip.

**Before running:**
- GPU on (Kaggle: Settings -> Accelerator -> GPU T4/P100). Internet on (for the dataset download).
- `best_finetuned_32frames.pt` must be available. Cell 2 searches for it automatically
  (`/kaggle/working`, `/kaggle/input`, current folder). If it is not found, upload it
  (Kaggle: Add Input -> upload as a dataset) or set `FINETUNED_CKPT` manually in Cell 2.

Run every cell in order, top to bottom.

Each exported sample has: `montage.png` (8 frames with true/predicted label), `audio.wav`,
`video.mp4` (rebuilt from the 32 frames + audio, because the dataset has no original mp4s), and
`summary.csv` lists all 30 with class probabilities.

## 1. Install dependencies

In [6]:
!pip install -q torchlibrosa kagglehub torchaudio torchvision

## 2. Imports, device, checkpoint location

In [7]:
import os, json, shutil, subprocess, random, tempfile, glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchvision
import torchvision.transforms as T

from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| torchaudio", torchaudio.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU found. Inference will be very slow on CPU.")

CKPT_NAME = "best_finetuned_32frames.pt"

# Set this manually if the automatic search below does not find your checkpoint:
FINETUNED_CKPT = None
# FINETUNED_CKPT = "/kaggle/input/<your-dataset-name>/best_finetuned_32frames.pt"
# FINETUNED_CKPT = "/workspace/best_finetuned_32frames.pt"   # vast.ai


def find_file(name, roots, max_depth=5, skip=("TikHarm_frames_32", "TikHarm_audio")):
    for root in roots:
        if not os.path.isdir(root):
            continue
        base_depth = root.rstrip("/").count("/")
        for dirpath, dirnames, filenames in os.walk(root):
            # do not descend into the (huge) frame / audio folders
            dirnames[:] = [d for d in dirnames if d not in skip]
            if dirpath.count("/") - base_depth >= max_depth:
                dirnames[:] = []
            if name in filenames:
                return os.path.join(dirpath, name)
    return None


if FINETUNED_CKPT is None:
    FINETUNED_CKPT = find_file(
        CKPT_NAME,
        ["/kaggle/working", "/kaggle/input", os.getcwd(), "/workspace", os.path.expanduser("~")],
    )

assert FINETUNED_CKPT and os.path.isfile(FINETUNED_CKPT), (
    f"Could not find {CKPT_NAME}. Upload it and set FINETUNED_CKPT in this cell."
)
print("Checkpoint:", FINETUNED_CKPT, f"({os.path.getsize(FINETUNED_CKPT) / 1e6:.1f} MB)")

torch 2.14.0+cu130 | torchvision 0.29.1+cu130 | torchaudio 2.11.0+cu130
Device: cpu
Checkpoint: /home/arsalan/Desktop/Multimodal_Major_Project/notebooks/kaggle/fusion32framesunfreezed/best_finetuned_32frames.pt (139.2 MB)


## 3. Dataset download and paths

In [8]:
import kagglehub

# On Kaggle with the dataset attached (Add Input), this returns the /kaggle/input path without downloading.
# Anywhere else it downloads the dataset (public, usually no credentials needed).
BASE = kagglehub.dataset_download("aryansraut/tikharm-data-32-frames")
print("Dataset path:", BASE)
print("Contents:", os.listdir(BASE))

NUM_FRAMES = 32
NUM_CLASSES = 4
MAX_AUDIO = 16000 * 10
BATCH_SIZE = 8
NUM_WORKERS = 2

CLASSES = ["Adult Content", "Harmful Content", "Safe", "Suicide"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}


def pick_root(candidates):
    for c in candidates:
        if os.path.isdir(os.path.join(c, "test")):
            return c
    raise FileNotFoundError(f"None of these contain a 'test' folder: {candidates}")


VISUAL_ROOT = pick_root([f"{BASE}/TikHarm_frames_32/TikHarm_frames_32", f"{BASE}/TikHarm_frames_32"])
AUDIO_ROOT  = pick_root([f"{BASE}/TikHarm_audio/TikHarm_audio", f"{BASE}/TikHarm_audio"])
print("Visual root:", VISUAL_ROOT)
print("Audio root: ", AUDIO_ROOT)

for cls in CLASSES:
    assert os.path.isdir(os.path.join(VISUAL_ROOT, "test", cls)), f"Missing frames: test/{cls}"
    assert os.path.isdir(os.path.join(AUDIO_ROOT, "test", cls)), f"Missing audio: test/{cls}"
print("Test videos per class:",
      {c: len(os.listdir(os.path.join(VISUAL_ROOT, "test", c))) for c in CLASSES})

  8%|█████▏                                                              | 990M/12.8G [01:07<13:44, 15.3MB/s]


KeyboardInterrupt: 

## 4. Audio loading helper and CNN14 (audio model)

In [ ]:
from torchlibrosa.stft import Spectrogram, LogmelFilterBank


def load_wav(path):
    """Returns (mono waveform tensor [N], sample_rate). Falls back to soundfile if torchaudio.load fails."""
    try:
        wav, sr = torchaudio.load(path)
        return wav.mean(dim=0), sr
    except Exception:
        import soundfile as sf
        data, sr = sf.read(path, dtype="float32", always_2d=True)
        return torch.from_numpy(data.T).mean(dim=0), sr


class CNN14(nn.Module):
    def __init__(self, classes_num=4):
        super().__init__()
        self.spectrogram_extractor = Spectrogram(
            n_fft=1024, hop_length=320, win_length=1024,
            window="hann", center=True, pad_mode="reflect"
        )
        self.logmel_extractor = LogmelFilterBank(
            sr=16000, n_fft=1024, n_mels=64, fmin=50, fmax=8000
        )
        self.bn0 = nn.BatchNorm2d(64)
        self.conv_block1 = self._conv_block(1, 64)
        self.conv_block2 = self._conv_block(64, 128)
        self.conv_block3 = self._conv_block(128, 256)
        self.conv_block4 = self._conv_block(256, 512)
        self.fc1 = nn.Linear(512, 512)
        self.fc_out = nn.Linear(512, classes_num)

    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)
        )

    def forward(self, x):
        x = self.spectrogram_extractor(x)
        x = self.logmel_extractor(x)
        x = x.transpose(1, 3)
        x = self.bn0(x)
        x = x.transpose(1, 3)
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        x = torch.mean(x, dim=3)
        x = torch.mean(x, dim=2)
        x = F.relu(self.fc1(x))
        return self.fc_out(x)

## 5. Dataset (same preprocessing as training)

In [ ]:
def list_frames(video_dir):
    return sorted(f for f in os.listdir(video_dir) if f.lower().endswith((".jpg", ".jpeg", ".png")))


def select_frames(frame_files, num_frames=NUM_FRAMES):
    if len(frame_files) >= num_frames:
        idx = torch.linspace(0, len(frame_files) - 1, num_frames).long()
        return [frame_files[i] for i in idx]
    return frame_files + [frame_files[-1]] * (num_frames - len(frame_files))


class FusionDataset(Dataset):
    def __init__(self, visual_root, audio_root, split, num_frames=32, max_audio=16000 * 10):
        self.samples = []
        self.num_frames = num_frames
        self.max_audio = max_audio
        self.transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        for cls in CLASSES:
            visual_class_dir = os.path.join(visual_root, split, cls)
            audio_class_dir = os.path.join(audio_root, split, cls)
            if not os.path.isdir(visual_class_dir):
                raise FileNotFoundError(f"Missing directory: {visual_class_dir}")

            for video_id in sorted(os.listdir(visual_class_dir)):
                video_dir = os.path.join(visual_class_dir, video_id)
                if not os.path.isdir(video_dir):
                    continue
                audio_path = os.path.join(audio_class_dir, video_id + ".wav")
                if os.path.isfile(audio_path):
                    self.samples.append((video_dir, audio_path, CLASS_TO_IDX[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        video_dir, audio_path, label = self.samples[index]

        frame_files = list_frames(video_dir)
        if len(frame_files) == 0:
            raise RuntimeError(f"No frames found: {video_dir}")

        frames = [
            self.transform(Image.open(os.path.join(video_dir, f)).convert("RGB"))
            for f in select_frames(frame_files, self.num_frames)
        ]
        video = torch.stack(frames).permute(1, 0, 2, 3)   # C, T, H, W

        waveform, sample_rate = load_wav(audio_path)
        if sample_rate != 16000:
            waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
        if waveform.shape[0] < self.max_audio:
            waveform = F.pad(waveform, (0, self.max_audio - waveform.shape[0]))
        else:
            waveform = waveform[:self.max_audio]

        return video, waveform, label


test_dataset = FusionDataset(VISUAL_ROOT, AUDIO_ROOT, "test", NUM_FRAMES, MAX_AUDIO)
print("Test samples:", len(test_dataset))

v, a, l = test_dataset[0]
assert v.shape == (3, NUM_FRAMES, 224, 224), v.shape
assert a.shape[0] == MAX_AUDIO, a.shape
print("Sample check OK | video", tuple(v.shape), "| audio", tuple(a.shape), "| label", CLASSES[l])

## 6. Load the fine-tuned model (`best_finetuned_32frames.pt`)

In [ ]:
class FeatureFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(768 + 512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 4)
        )

    def forward(self, visual_features, audio_features):
        return self.network(torch.cat([visual_features, audio_features], dim=1))


def build_swin3d():
    # Local torchvision first (no internet needed); fall back to the hub version used in training.
    try:
        from torchvision.models.video import swin3d_t
        m = swin3d_t(weights=None)
    except Exception:
        m = torch.hub.load("pytorch/vision:v0.15.2", "swin3d_t", pretrained=False)
    m.head = nn.Linear(m.head.in_features, NUM_CLASSES)
    return m


ckpt = torch.load(FINETUNED_CKPT, map_location="cpu")
print("Checkpoint keys:", list(ckpt.keys()))

visual_model = build_swin3d()
try:
    visual_model.load_state_dict(ckpt["visual_state_dict"])
except RuntimeError:
    print("Local swin3d_t keys did not match, retrying with torch.hub v0.15.2 ...")
    visual_model = torch.hub.load("pytorch/vision:v0.15.2", "swin3d_t", pretrained=False)
    visual_model.head = nn.Linear(visual_model.head.in_features, NUM_CLASSES)
    visual_model.load_state_dict(ckpt["visual_state_dict"])

audio_model = CNN14(NUM_CLASSES)
audio_model.load_state_dict(ckpt["audio_state_dict"])

fusion_model = FeatureFusion()
fusion_model.load_state_dict(ckpt["fusion_state_dict"])

for m in (visual_model, audio_model, fusion_model):
    m.to(device).eval()
    for p in m.parameters():
        p.requires_grad = False

del ckpt


def get_visual_features(x):
    x = visual_model.patch_embed(x)
    x = visual_model.pos_drop(x)
    for layer in visual_model.features:
        x = layer(x)
    x = visual_model.norm(x)
    return x.mean(dim=(1, 2, 3))          # B, T, H, W, C -> B, C


def get_audio_features(x):
    # STFT / log-mel kept in fp32 (fp16 here caused NaNs during training)
    with torch.autocast("cuda", enabled=False):
        x = x.float()
        x = audio_model.spectrogram_extractor(x)
        x = audio_model.logmel_extractor(x)
    x = x.transpose(1, 3)
    x = audio_model.bn0(x)
    x = x.transpose(1, 3)
    x = audio_model.conv_block1(x)
    x = audio_model.conv_block2(x)
    x = audio_model.conv_block3(x)
    x = audio_model.conv_block4(x)
    x = x.mean(dim=3)
    x = x.mean(dim=2)
    return F.relu(audio_model.fc1(x))


print("Fine-tuned model loaded on", device)

## 7. Inference on the full test split

In [ ]:
infer_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,                 # keeps predictions aligned with test_dataset.samples
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

USE_AMP = torch.cuda.is_available()
all_probs = []

with torch.no_grad():
    for videos, audios, _ in tqdm(infer_loader, desc="Inference"):
        videos = videos.to(device, non_blocking=True)
        audios = audios.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = fusion_model(get_visual_features(videos), get_audio_features(audios))
        all_probs.append(F.softmax(logits.float(), dim=1).cpu())

probs = torch.cat(all_probs).numpy()
assert np.isfinite(probs).all(), "Non-finite predictions. Set USE_AMP = False and re-run this cell."

y_pred = probs.argmax(1)
conf = probs.max(1)
y_true = np.array([s[2] for s in test_dataset.samples])

from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

print(f"Accuracy:          {accuracy_score(y_true, y_pred):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")
print(f"Correct: {(y_pred == y_true).sum()} | Incorrect: {(y_pred != y_true).sum()}")
print()
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))
print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(confusion_matrix(y_true, y_pred), index=CLASSES, columns=CLASSES))

## 8. Select 20 correct and 10 incorrect samples

In [ ]:
SEED = 42
N_CORRECT, N_WRONG = 20, 10
rng = random.Random(SEED)

correct_idx = [i for i in range(len(y_true)) if y_pred[i] == y_true[i]]
wrong_idx   = [i for i in range(len(y_true)) if y_pred[i] != y_true[i]]

# 20 correct: round-robin across the 4 true classes so every class is represented (5 each if possible)
by_class = {c: [i for i in correct_idx if y_true[i] == c] for c in range(NUM_CLASSES)}
for lst in by_class.values():
    rng.shuffle(lst)

sel_correct = []
target = min(N_CORRECT, len(correct_idx))
while len(sel_correct) < target:
    for c in range(NUM_CLASSES):
        if by_class[c] and len(sel_correct) < target:
            sel_correct.append(by_class[c].pop())

# 10 incorrect: random
wrong_shuffled = wrong_idx[:]
rng.shuffle(wrong_shuffled)
sel_wrong = wrong_shuffled[:N_WRONG]

# Alternatives (uncomment to use):
# sel_correct = sorted(correct_idx, key=lambda i: -conf[i])[:N_CORRECT]   # most confident correct ones
# sel_wrong   = sorted(wrong_idx,   key=lambda i: -conf[i])[:N_WRONG]     # most confident mistakes

if len(sel_correct) < N_CORRECT:
    print(f"Only {len(sel_correct)} correct samples available.")
if len(sel_wrong) < N_WRONG:
    print(f"Only {len(sel_wrong)} incorrect samples available.")

print("Correct selected per class:",
      {CLASSES[c]: int(sum(y_true[i] == c for i in sel_correct)) for c in range(NUM_CLASSES)})
print("Incorrect selected (true -> pred):",
      [f"{CLASSES[y_true[i]]} -> {CLASSES[y_pred[i]]}" for i in sel_wrong])

## 9. Export samples (montage + audio + mp4 + CSV) and zip

In [ ]:
OUT_ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
OUT_DIR = os.path.join(OUT_ROOT, "demo_samples")
shutil.rmtree(OUT_DIR, ignore_errors=True)
os.makedirs(OUT_DIR)

MAKE_MP4 = shutil.which("ffmpeg") is not None
print("ffmpeg available:", MAKE_MP4, "" if MAKE_MP4 else "(mp4s will be skipped; montage + wav still exported)")

try:
    FONT = ImageFont.load_default(size=18)
except TypeError:
    FONT = ImageFont.load_default()


def make_montage(video_dir, header, out_path, cols=4, rows=2, thumb_h=240):
    chosen = select_frames(list_frames(video_dir))
    step = max(1, len(chosen) // (cols * rows))
    chosen = chosen[::step][:cols * rows]
    imgs = [Image.open(os.path.join(video_dir, f)).convert("RGB") for f in chosen]
    w0, h0 = imgs[0].size
    tw, th = max(1, int(w0 * thumb_h / h0)), thumb_h
    banner = 40
    canvas = Image.new("RGB", (tw * cols, th * rows + banner), (20, 20, 20))
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 10), header, fill=(255, 255, 255), font=FONT)
    for k, im in enumerate(imgs):
        canvas.paste(im.resize((tw, th)), ((k % cols) * tw, banner + (k // cols) * th))
    canvas.save(out_path)


def wav_duration(path):
    try:
        import soundfile as sf
        return sf.info(path).duration
    except Exception:
        wav, sr = load_wav(path)
        return wav.shape[0] / sr


def make_mp4(video_dir, wav_path, out_path):
    duration = min(max(wav_duration(wav_path), 1.0), MAX_AUDIO / 16000)
    fps = NUM_FRAMES / duration
    with tempfile.TemporaryDirectory() as tmp:
        for k, f in enumerate(select_frames(list_frames(video_dir)), 1):
            Image.open(os.path.join(video_dir, f)).convert("RGB").save(os.path.join(tmp, f"{k:04d}.jpg"))
        cmd = [
            "ffmpeg", "-y", "-loglevel", "error",
            "-framerate", f"{fps:.4f}", "-i", os.path.join(tmp, "%04d.jpg"),
            "-i", wav_path,
            "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
            "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac", "-shortest",
            out_path,
        ]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print("ffmpeg failed:", r.stderr[:300])
            return False
    return True


records = []
for group, idxs in (("correct", sel_correct), ("incorrect", sel_wrong)):
    for k, i in enumerate(tqdm(idxs, desc=f"Export {group}"), 1):
        video_dir, audio_path, _ = test_dataset.samples[i]
        vid = os.path.basename(video_dir)
        t_name, p_name = CLASSES[y_true[i]], CLASSES[y_pred[i]]

        folder = os.path.join(
            OUT_DIR, group,
            f"{k:02d}_true-{t_name.replace(' ', '')}_pred-{p_name.replace(' ', '')}_{conf[i] * 100:.0f}pct",
        )
        os.makedirs(folder)

        header = f"TRUE: {t_name}   PRED: {p_name}   CONF: {conf[i] * 100:.1f}%"
        make_montage(video_dir, header, os.path.join(folder, "montage.png"))
        shutil.copy(audio_path, os.path.join(folder, "audio.wav"))
        mp4_ok = make_mp4(video_dir, audio_path, os.path.join(folder, "video.mp4")) if MAKE_MP4 else False

        rec = dict(
            group=group, folder=os.path.relpath(folder, OUT_DIR), video_id=vid,
            true=t_name, pred=p_name, confidence=round(float(conf[i]), 4), mp4=mp4_ok,
        )
        rec.update({f"p_{c}": round(float(probs[i][j]), 4) for j, c in enumerate(CLASSES)})
        records.append(rec)

df = pd.DataFrame(records)
df.to_csv(os.path.join(OUT_DIR, "summary.csv"), index=False)

zip_path = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("Exported", len(df), "samples")
print("Zip:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")

## 10. Preview and download

In [ ]:
from IPython.display import display, FileLink, Image as IPImage, Video as IPVideo

display(df[["group", "video_id", "true", "pred", "confidence"]])

# Inline preview: first 3 of each group
for group in ("correct", "incorrect"):
    print(f"\n========== {group.upper()} ==========")
    for folder in df[df.group == group].folder.head(3):
        display(IPImage(os.path.join(OUT_DIR, folder, "montage.png")))

# Optional: play one exported video inline
# display(IPVideo(os.path.join(OUT_DIR, df.folder.iloc[0], "video.mp4"), embed=True, width=480))

# Clickable link (works in Jupyter / vast.ai; on Kaggle use the Output panel instead)
try:
    display(FileLink(os.path.relpath(zip_path)))
except Exception:
    pass
print("Zip saved at:", zip_path)

## Downloading the zip

- **Kaggle:** right sidebar -> Output (`/kaggle/working`) -> three-dot menu on `demo_samples.zip` -> Download.
  Files in `/kaggle/working` are lost when the session ends unless you Save Version, so download right away.
- **vast.ai / Jupyter:** click the link printed by Cell 10, or right-click `demo_samples.zip` in the file browser -> Download.

## Troubleshooting

- `Could not find best_finetuned_32frames.pt` -> upload it and set `FINETUNED_CKPT` in Cell 2.
- Size-mismatch / key errors in Cell 6 -> the checkpoint is not the fine-tuned one (e.g. you pointed at
  `best_feature_fusion_32frames.pth`, which only has the fusion head).
- `Non-finite predictions` in Cell 7 -> set `USE_AMP = False` at the top of Cell 7 and re-run it.
- CUDA out of memory in Cell 7 -> lower `BATCH_SIZE` in Cell 3 (e.g. 4) and re-run Cells 3-7.